# **BERT: Overview and Training Process**  

BERT (Bidirectional Encoder Representations from Transformers) is a deep learning model designed for understanding language context using bidirectional learning. Unlike traditional models that process text sequentially, BERT analyzes words in relation to their surroundings, leading to deeper linguistic comprehension. Pretrained on large datasets, it can be fine-tuned for tasks like text classification and question answering.  

BERT’s training involves Masked Language Modeling (MLM) and Next Sentence Prediction (NSP). MLM masks words and trains BERT to predict them using context, improving sentence understanding. NSP helps recognize sentence relationships. After pretraining, fine-tuning adapts BERT to specific tasks, retaining its linguistic knowledge while refining it for targeted applications.

Installation Instructions:
This Colab notebook runs in a virtual Python environment that already includes many popular libraries by default. However, for some projects, you may need to install or update specific library versions.

Before running the demo below, please execute the installation cell first. When prompted with "Restart runtime?", click Cancel and wait for the installation to complete fully.

Once the installation finishes, manually restart the session by going to Runtime → Restart session.

After restarting, you can proceed to run the next cell, which imports the necessary libraries.

In [7]:
# Install specific versions of necessary libraries
%pip install datasets==3.2.0  #Installing the 'datasets' library (version 3.2.0)
%pip install faiss_cpu==1.10.0 #Installing the 'faiss_cpu' library (version 1.10.0)
%pip install numpy==1.26.4 #Installing the 'numpy' library (version 1.26.4)
%pip install sentence_transformers=3.0.1  #Installing 'sentence_transformers' (version 3.0.1) for sentence/text embeddings
%pip install torch==2.3.1 # Installing the 'torch' library (version 2.3.1), PyTorch, for deep learning
%pip install transformers==4.44.0 # Installing the 'transformers' library (version 4.44.0) from Hugging Face for pre-trained models


%pip install datasets==3.2.0  #Installing the 'datasets' library (version 3.2.0)
%pip install torch==2.3.1 # Installing the 'torch' library (version 2.3.1), PyTorch, for deep learning
%pip install transformers==5.0.0 # Installing the 'transformers' library (version 4.44.0) from Hugging Face for pre-trained models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.3/553.3 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.19.1
    Uninstalling tokenizers-0.19.1:
      Successfully uninstalled tokenizers-0.19.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.44.0
    Uninstalling transformers-4.44.0:
      Successfully uninstalled transformers-4.44.0


In [1]:
### Restart Session (Runtime -> Restart Session) before running ***

import random
import numpy as np
import torch
from transformers import BertTokenizer, BertForPreTraining, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset

In [2]:
# Load BERT tokenizer
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

# Load BERT Model
model = BertForPreTraining.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

## **BERT Masked Language Modeling (MLM)**
BERT is trained using Masked Language Modeling (MLM), where random words in a sentence are replaced with a [MASK] token. The model then predicts the masked words based on context, improving its understanding of language semantics.

### Steps in MLM
- Tokenization: Convert input text into tokens using a pretrained BERT tokenizer.
- Masking: Randomly replace words with the [MASK] token.
- Prediction: Use BERT to predict the masked words based on surrounding context.
- Loss Calculation: Compare predicted words with original words to optimize the model.

In [3]:
text = "Hello there! I am very [MASK] to meet you after so [MASK]!" # special token used for Masking Text

tokenised_input = tokenizer(text, return_tensors="pt") # embeddings of input text
tokenised_input["input_ids"]

tensor([[ 101, 7592, 2045,  999, 1045, 2572, 2200,  103, 2000, 3113, 2017, 2044,
         2061,  103,  999,  102]])

In [19]:
with torch.no_grad(): # without amending gradient weights/ tuning the model further
    outputs = model(**tokenised_input) # predictions are made based off tokenised input - for [MASK]

predictions = outputs.prediction_logits # Masked Language Model outputs - vector embeddings of MASK tokens are predicted
torch.set_printoptions(threshold=100)


In the Hugging Face transformers library, when using BertForPreTraining, the model outputs both prediction_logits and seq_relationship_logits. These correspond to the two core tasks BERT is trained on: Masked Language Modeling (MLM) and Next Sentence Prediction (NSP).
Here is a detailed breakdown of both:
1. prediction_logits (Masked Language Model Head)
- Definition: These are the raw, unnormalized output scores (logits) from the language modeling head for each token in the input sequence.
- Purpose: To predict the original token of masked input tokens ([MASK]).
- Shape: (batch_size, sequence_length, config.vocab_size).
- What it represents: For every position in your input sequence, it provides a score for every single word in the BERT vocabulary. A higher score means the model believes that specific vocabulary word is the likely masked token.
- Next Step: To get probabilities, you apply the Softmax function to these logits.
2. seq_relationship_logits (Next Sentence Prediction Head)
- Definition: These are the raw, unnormalized output scores from the next sentence prediction (classification) head.
- Purpose: To determine if the second sentence in the input pair (Sentence B) naturally follows the first sentence (Sentence A).
- Shape: (batch_size, 2).
- What it represents: It outputs two values:
  - Index 0: The probability that sentence B is the direct continuation of sentence A (True label).
  - Index 1: The probability that sentence B is a random sentence from the corpus (False label).
- Next Step: Similar to prediction_logits, you apply Softmax to get the probability of whether the relationship is True or False.

In [20]:
masked_index = (tokenised_input["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1] # get indexes with masked tokens
print(masked_index) # can refer back to tokenised_input results - value of 103 is accorded to [MASK]

for idx in masked_index:
    predicted_token_id = predictions[0, idx].argmax(dim=-1) # get indexes of the masked tokens and checks prediction results
    predicted_token = f"**{tokenizer.decode(predicted_token_id).replace(' ','')}**"
    text = text.replace(tokenizer.mask_token, predicted_token,1) # final 1 is to replace tokens 1 at a time - processed individually
print(text)

tensor([ 7, 13])
Hello there! I am very **pleased** to meet you after so **long**!


In [21]:
### compiled as a function
def MLM_prediction(input_text,n_masks=0):
    tokenised_input = tokenizer(input_text, return_tensors="pt")

    original_tokens = tokenizer.convert_ids_to_tokens(tokenised_input["input_ids"].squeeze(0))
    mask_idx = random.sample(range(1,tokenised_input["input_ids"].size()[-1]-1),n_masks)
    new_input_id = [v if i not in mask_idx else 103 for i, v in enumerate(np.array(tokenised_input["input_ids"].reshape(1,-1)[0]))]
    tokenised_input["input_ids"] = torch.tensor(np.array(new_input_id)).reshape(1,-1)
    masked_tokens = original_tokens[:]

    with torch.no_grad():
        outputs = model(**tokenised_input)
    predictions = outputs.prediction_logits
    masked_index = (tokenised_input["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    for idx in masked_index:
        predicted_token_id = predictions[0, idx].argmax(dim=-1)
        predicted_token = f"**{tokenizer.decode(predicted_token_id).replace(' ','')}**"
        masked_tokens[idx] = predicted_token
    reconstructed_tokens = tokenizer.convert_tokens_to_string(masked_tokens)
    return reconstructed_tokens

In [22]:
random.seed(30) # can remove/ change the number for more instances
my_input = """In recent years, advancements in artificial intelligence and machine learning have revolutionized various industries,
including healthcare, finance, and autonomous driving, by enabling predictive analytics, automated decision-making,
and real-time data processing"""
MLM_prediction(my_input, n_masks = 8)

'[CLS] in **recent** years **,** advancements in artificial intelligence and machine learning have **revolution**ized various **industries**, **including** healthcare, finance, and autonomous driving, by enabling predictive analytics, automated **decision** - making, and **multi** - **based** data processing [SEP]'

In [ ]:
torch.tensor(np.array([v if i not in [] else 103 for i, v in enumerate(np.array(tokenised_input["input_ids"].reshape(1,-1)[0]))])).reshape(1,-1)

tensor([[ 101, 7592, 2045,  999, 1045, 2572, 2200,  103, 2000, 3113, 2017, 2044,
         2061,  103,  999,  102]])

## **BERT Next Sentence Prediction (NSP)**
In addition to MLM, BERT is trained with Next Sentence Prediction (NSP) to improve its understanding of sentence relationships. This helps in tasks like question answering and coherent text generation.

### Steps in NSP
- Sentence Pair Selection - 2 sentences where:
    - 50% of the time, the second sentence follows the first logically.
    - 50% of the time, the second sentence is unrelated.
- Tokenization: Convert sentences into tokenized representations.
- Model Processing: BERT classifies whether the second sentence follows the first.
- Loss Calculation: Optimize BERT’s classification ability using labeled sentence pairs.

In [4]:
sen_1 = "The capital of the nation of Italy is Rome."
sen_2 = "Do you believe in the force of gravity?"

tokenised_inputs = tokenizer(sen_1, sen_2, return_tensors="pt")
tokenised_inputs["input_ids"]

tensor([[ 101, 1996, 3007, 1997, 1996, 3842, 1997, 3304, 2003, 4199, 1012,  102,
         2079, 2017, 2903, 1999, 1996, 2486, 1997, 8992, 1029,  102]])

In [5]:
with torch.no_grad():
    outputs = model(**tokenised_inputs)

nsp_logits = outputs.seq_relationship_logits
nsp_logits # logits for if sentences are CONSECUTIVE or NOT CONSECUTIVE respectively


tensor([[-2.1036,  4.8527]])

In [6]:
pred_label = torch.argmax(torch.nn.functional.softmax(nsp_logits, dim=-1))
result = "Coherent Sentences" if pred_label == 0 else "Discontinuous sentences"
print(result)

Discontinuous sentences


In [3]:
def is_next_sentence_related(sentence1, sentence2):
    # Tokenize the sentences
    encoding = tokenizer(sentence1, sentence2, return_tensors='pt', padding=True, truncation=True)

    # Get the prediction from the model
    with torch.no_grad():
        outputs = model(**encoding)
    logits = outputs.seq_relationship_logits

    # If the logits indicate that sentence2 is the next sentence
    return torch.argmax(logits) == 0

def group_sentences_with_nsp(paragraph):
    import re
    doc = re.split(r'[.?!]\s',paragraph) # very simple sentence splitter
    sentences = [sent.strip() for sent in doc]  # Split paragraph into sentences

    # Group sentences
    grouped_sentences = []
    current_group = [sentences[0]]

    for i in range(1, len(sentences)):
        # Check if the current sentence should follow the previous one using BERT NSP
        related = is_next_sentence_related(sentences[i-1], sentences[i])
        if related:
            # If sentences are related, add to the current group
            current_group.append(sentences[i])
        else:
            # If sentences are unrelated, start a new group
            grouped_sentences.append(current_group)
            current_group = [sentences[i]]

    # Add the last group
    grouped_sentences.append(current_group)

    return grouped_sentences

In [4]:
para = """
The sky is bright and blue. It is a nice day outside. The trees are swaying in the wind. Tommy was hacked yesterday by some unknown person.
I heard he lost quite a bit of money. He has made a police report accordingly. The beach at Sicily was a soo beautiful!
I spent a lot of time there - it was very peaceful indeed. I would love to go back someday!"""

grouped_sentences = group_sentences_with_nsp(para)
for group in grouped_sentences:
    print("Group:")
    for sentence in group:
        print(f" - {sentence}")

Group:
 - The sky is bright and blue
 - It is a nice day outside
 - The trees are swaying in the wind
Group:
 - Tommy was hacked yesterday by some unknown person
 - I heard he lost quite a bit of money
 - He has made a police report accordingly
Group:
 - The beach at Sicily was a soo beautiful
 - I spent a lot of time there - it was very peaceful indeed
 - I would love to go back someday!
